# Knockout Absorption Analysis — Kalshi World Cup markets

**Question:** when a team is eliminated, where does its championship probability go — to the team that knocked it out, to the winner's future opponents, or to the field?

**Why it matters:** the `loser-champ + winner-advances` hedge only pays if the surviving longshot's championship contract reprices by (at least) the consistency-implied amount, and you can exit there. The `excess_absorption` ratio measured here is the key empirical input.

**Prereqs:** run `python -m src.discover --draft-matches`, fill `src/config.py`, then `python -m src.pull_data`. That produces the two CSVs loaded below.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from src.decompose import run_all, summarize, hedge_pnl
from src.plotting import absorption_bars, hedge_frontier

snapshots = pd.read_csv('../data/processed/snapshots.csv')
matches   = pd.read_csv('../data/processed/matches.csv')
matches

## 1. Sanity checks
Raw prices carry vig; the decomposition normalizes internally, but check the raw books sum to something sane (1.02–1.15 typical) and that no snapshot is missing.

In [ ]:
chk = snapshots.groupby('match_id')[['pre','post']].sum()
print(chk)
missing = snapshots[snapshots[['pre','post']].isna().any(axis=1)]
print(f"\nmissing snapshots: {len(missing)}")
missing

## 2. Per-match decomposition

Key columns:
- `winner_share` — fraction of the loser's released probability the winner actually absorbed (can exceed 1: the winner may also drain probability from future opponents whose path got harder)
- `implied_winner_share` — what joint coherence of match + champ markets predicts
- `excess_absorption` — actual / implied. **>1**: winners get over-rewarded → the hedge has extra edge. **<1**: mass leaks to the field → the naive hedge math overstates the payoff.

In [ ]:
results = run_all(snapshots, matches)
results.round(3)

In [ ]:
summarize(results)

In [ ]:
absorption_bars(results, 'absorption_decomposition.png')
from IPython.display import Image; Image('absorption_decomposition.png')

## 3. Cuts worth looking at
- **By favorite status**: does excess absorption differ when the favorite advances vs. the underdog? (The hedge scenario cares about the *underdog-advances* branch.)
- **By loser size**: tiny `loser_pre` means a tiny denominator — noisy estimates. Weight or filter.
- **Vig drift**: if `raw_vig_post - raw_vig_pre` is large, the snapshot timing is picking up book-wide repricing, not match-specific flow. Consider tightening `POST_WINDOW_S` or using VWAP over a few candles.

In [ ]:
results['underdog_won'] = results.q_winner < 0.5
results.groupby('underdog_won')['excess_absorption'].describe()

## 4. Hedge P&L under the *measured* excess absorption

Your live position: 1,000 BEL champ YES @ 1.28¢; BEL trading 1.4¢; USA-advances 54¢, q(USA)=0.53. Swap in current prices before trusting the window.

In [ ]:
ea = results.excess_absorption.median()
pnl = hedge_pnl(n_loser_contracts=1000, loser_entry_c=1.28, loser_pre_c=1.4,
                q_winner=0.53, hedge_price_c=54.0, excess_absorption=ea)
hedge_frontier(pnl, 'hedge_frontier.png',
               f'BEL champ + USA-advances hedge (excess absorption={ea:.2f})')
print(pnl[pnl.worst_case > 0].to_string(index=False))
from IPython.display import Image; Image('hedge_frontier.png')

## 5. Open questions for iteration
1. **Exit liquidity**: the repricing only monetizes if you can sell 1,000 BEL near the new mid. Pull order-book depth (authenticated endpoint) around past repricings.
2. **Speed**: how fast does the champ market reprice after the whistle vs. after Kalshi settles the match market? If slow, there may be a latency trade *instead of* a hedge.
3. **Symmetry**: `hedge_pnl` assumes the same absorption dynamics when the underdog advances. Section 3's underdog cut tests that directly.